# B05 · Sesión 5 — Lógica difusa y dinámica

**Objetivo (RA5-b/c/d):** aplicar lógica difusa con `scikit-fuzzy` (fuzzificación → reglas → desfuzzificación) y analizar sensibilidad/robustez.

> Práctica guiada de la Sesión 5 de los [apuntes](../apuntes.md).

In [ ]:
%pip install scikit-fuzzy

## 1. El problema de la propina (ejemplo canónico)

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

servicio = ctrl.Antecedent(np.arange(0, 11, 0.1), "servicio")
comida = ctrl.Antecedent(np.arange(0, 11, 0.1), "comida")
propina = ctrl.Consequent(np.arange(0, 26, 0.1), "propina")

servicio["baja"] = fuzz.trimf(servicio.universe, [0, 0, 5])
servicio["media"] = fuzz.trimf(servicio.universe, [0, 5, 10])
servicio["alta"] = fuzz.trimf(servicio.universe, [5, 10, 10])
comida.automf(3)
propina["baja"] = fuzz.trimf(propina.universe, [0, 0, 13])
propina["media"] = fuzz.trimf(propina.universe, [0, 13, 25])
propina["alta"] = fuzz.trimf(propina.universe, [13, 25, 25])

r1 = ctrl.Rule(servicio["baja"] | comida["poor"], propina["baja"])
r2 = ctrl.Rule(servicio["media"], propina["media"])
r3 = ctrl.Rule(servicio["alta"] | comida["good"], propina["alta"])

sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem([r1, r2, r3]))
sim.input["servicio"] = 9.8
sim.input["comida"] = 6.5
sim.compute()
print(f"Propina: {sim.output['propina']:.2f} EUR")   # ~19,24

## 2. Actividad — control difuso de riego

Dos entradas (`temperatura`, `humedad`) y una salida (`riego`). Define funciones de pertenencia y 3 reglas: a más temperatura y menos humedad, más riego.

In [ ]:
temperatura = ctrl.Antecedent(np.arange(0, 41, 0.5), "temperatura")
humedad = ctrl.Antecedent(np.arange(0, 101, 1), "humedad")
riego = ctrl.Consequent(np.arange(0, 11, 0.5), "riego")

# TODO: funciones de pertenencia (baja/media/alta) y reglas
...

## 3. Sensibilidad: variar un umbral

Cambia un punto de una función de pertenencia y observa cómo se desplaza la salida. Un sistema muy sensible detecta antes, pero da más **falsas alarmas**.

In [ ]:
# Compara la salida con el triangulo "alta" de servicio en [5,10,10] y en [6,10,10]
# TODO: reconstruye el sistema con el nuevo triangulo y compara la propina para servicio=7
...

## 4. Histéresis (concepto)

Un sensor de CO que oscila 28-32 ppm con umbral 30 enciende/apaga extractores sin parar. Solución: exigir que la alerta se mantenga 30 s, o suavizar con un controlador difuso.

**Para casa:** explica en 3 líneas la diferencia entre **sensibilidad** y **robustez**.